# Databricks Notebook for ODI to Spark SQL Migration

**Source File:** `undefined`
**Conversion Timestamp:** 2024-07-29 12:00:00 UTC

**Description:** This notebook migrates an ODI process for `WC_BADGE_DETAILS_D` from Oracle to Databricks Spark SQL. It involves staging, flow processing, and merging data into the target dimension table. Key transformations include handling Oracle data types, function conversions, and rewriting ODI's specific deduplication and MERGE patterns for optimal Spark SQL performance and correctness.

In [ ]:
dbutils.widgets.text("ETL_JOB_TYPE", "", "1. ETL Job Type")
dbutils.widgets.text("DATASOURCE_NUM_ID", "380", "2. Data Source Number ID")
dbutils.widgets.text("ETL_PROC_WID", "", "3. ETL Process WID")
dbutils.widgets.text("ODI_SESS_NO", "", "4. ODI Session Number")

## ETL Parameters

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {2}: Get ETL last extract time
CREATE OR REPLACE TEMPORARY VIEW v_etl_last_extract_time AS
SELECT
    MAX(etl_last_extract_time) AS etl_last_extract_time
FROM workspace.prxbi_dw.wc_etl_parameters
WHERE ETL_JOB_TYPE = '${ETL_JOB_TYPE}';

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {3}: Get ETL current extract time
CREATE OR REPLACE TEMPORARY VIEW v_etl_current_extract_time AS
SELECT
    MAX(etl_current_extract_time) AS etl_current_extract_time
FROM workspace.prxbi_dw.wc_etl_parameters
WHERE ETL_JOB_TYPE = '${ETL_JOB_TYPE}';

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {6}: Get ETL ROW_WID
CREATE OR REPLACE TEMPORARY VIEW v_row_wid AS
SELECT
    MAX(ROW_WID) AS row_wid
FROM workspace.prxbi_dw.wc_etl_parameters
WHERE ETL_JOB_TYPE = '${ETL_JOB_TYPE}';

In [ ]:
display(spark.sql("SELECT '${ETL_JOB_TYPE}' AS ETL_JOB_TYPE, ${DATASOURCE_NUM_ID} AS DATASOURCE_NUM_ID, '${ETL_PROC_WID}' AS ETL_PROC_WID, '${ODI_SESS_NO}' AS ODI_SESS_NO"))
display(spark.sql("SELECT etl_last_extract_time
FROM v_etl_last_extract_time"))
display(spark.sql("SELECT etl_current_extract_time
FROM v_etl_current_extract_time"))
display(spark.sql("SELECT row_wid
FROM v_row_wid"))

## Staging Table (`C$_`)

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {30}:
Drop staging table
DROP TABLE IF EXISTS workspace.prxbi_dw.c_badge_details_stg;

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {40}: Create staging table
CREATE TABLE workspace.prxbi_dw.c_badge_details_stg
(
	ID	STRING,
	BADGELOCATION	STRING,
	BADGETOKEN	STRING,
	BADGEVERSION	SMALLINT,
	CONTACTEMAIL	STRING,
	CONTACTFIRSTNAME	STRING,
	CONTACTJOBTITLE	STRING,
	CONTACTLASTNAME	STRING,
	CONTACTPERSONRXMASTERID	STRING,
	CREATEDBYREGISTRATIONTYPE	STRING,
	CREATEDBYTYPE	STRING,
	CULTURE	STRING,
	CUSTOMERTYPE	STRING,
	EVENTEDITIONGBSCODE	STRING,
	ISBADGEUPDATE	STRING,
	MARKETINGPREFERENCESPROMPTREQU	STRING,
	ORGANISATIONCITY	STRING,
	ORGANISATIONCOUNTRYCODE	STRING,
	ORGANISATIONDISPLAYNAME	STRING,
	ORGANISATIONRXMASTERID	STRING,
	ORGANISATIONSTATE	STRING,
	PARTICIPATINGORGANISATIONID	STRING,
	PRODUCTCODE	STRING,
	QRCODECONTENT	STRING,
	REGISTRATIONID	STRING,
	STATUS	BIGINT,
	SUPPORTSTAFFCOMPANYADDRESS	STRING,
	SUPPORTSTAFFCOMPANYNAME	STRING,
	SUPPORTSTAFFMOBILEPHONE	STRING,
	SUPPORTSTAFFREPORTSTO	STRING,
	SUPPORTSTAFFSTANDS	STRING,
	SUPPORTSTAFFUSERACCESS	STRING,
	VERSIONNUMBER	BIGINT,
	MOBILEPHONE	STRING,
	FIRSTSCANNEDDATE	TIMESTAMP,
	LASTPRINTEDDATE	TIMESTAMP,
	ACCESSVALIDITYMODIFIEDDATE	TIMESTAMP,
	CREATEDDATE	TIMESTAMP,
	COMPANYPRODUCTCODE	STRING,
	PAYMENTSTATUS	STRING,
	PHOTOKEY	STRING,
	PHOTOSOURCE	STRING,
	PHOTOSOURCETYPE	STRING
)
USING DELTA;

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {50}: Insert into staging table with deduplication
-- Converted ODI MAX self-join dedup to ROW_NUMBER() window function
INSERT INTO workspace.prxbi_dw.c_badge_details_stg
SELECT
    ID,
    BADGELOCATION,
    BADGETOKEN,
    BADGEVERSION,
    CONTACTEMAIL,
    CONTACTFIRSTNAME,
    CONTACTJOBTITLE,
    CONTACTLASTNAME,
    CONTACTPERSONRXMASTERID,
    CREATEDBYREGISTRATIONTYPE,
    CREATEDBYTYPE,
    CULTURE,
    CUSTOMERTYPE,
    EVENTEDITIONGBSCODE,
    ISBADGEUPDATE,
    MARKETINGPREFERENCESPROMPTREQU,
    ORGANISATIONCITY,
    ORGANISATIONCOUNTRYCODE,
    ORGANISATIONDISPLAYNAME,
    ORGANISATIONRXMASTERID,
    ORGANISATIONSTATE,
    PARTICIPATINGORGANISATIONID,
    PRODUCTCODE,
    QRCODECONTENT,
    REGISTRATIONID,
    STATUS,
    SUPPORTSTAFFCOMPANYADDRESS,
    SUPPORTSTAFFCOMPANYNAME,
    SUPPORTSTAFFMOBILEPHONE,
    SUPPORTSTAFFREPORTSTO,
    SUPPORTSTAFFSTANDS,
    SUPPORTSTAFFUSERACCESS,
    VERSIONNUMBER,
    MOBILEPHONE,
    FIRSTSCANNEDDATE,
    LASTPRINTEDDATE,
    ACCESSVALIDITYMODIFIEDDATE,
    CREATEDDATE,
    COMPANYPRODUCTCODE,
    PAYMENTSTATUS,
    PHOTOKEY,
    PHOTOSOURCE,
    PHOTOSOURCETYPE
FROM (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY ID
            ORDER BY INT_INSERT_DATE DESC, VERSIONNUMBER DESC
        ) AS rn
    FROM workspace.prxbi_ts.wc_mercury_badge_ts
    WHERE
        INT_INSERT_DATE > (SELECT etl_last_extract_time FROM v_etl_last_extract_time)
        AND INT_INSERT_DATE <= (SELECT etl_current_extract_time FROM v_etl_current_extract_time)
) AS deduped_source
WHERE deduped_source.rn = 1;

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {60}: DBMS_STATS.GATHER_TABLE_STATS is not applicable for Delta staging tables, removed.
SELECT COUNT(*)
FROM workspace.prxbi_dw.c_badge_details_stg;

## Flow Table (`I$_`)

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {80}:
Drop flow table
DROP TABLE IF EXISTS workspace.prxbi_dw.i_wc_badge_details_d_flow;

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {90}: Create flow table
CREATE TABLE workspace.prxbi_dw.i_wc_badge_details_d_flow
(
	ROW_WID		BIGINT,
	BADGE_ID		STRING,
	BADGE_LOCATION		STRING,
	BADGE_TOKEN		STRING,
	BADGE_VERSION		SMALLINT,
	CONTACT_EMAIL		STRING,
	CONTACT_FIRST_NAME		STRING,
	CONTACT_LAST_NAME		STRING,
	CONTACT_JOB_TITLE		STRING,
	CONTACT_PERSON_ID		STRING,
	CREATION_REG_TYPE		STRING,
	CREATION_TYPE		STRING,
	CULTURE		STRING,
	CUSTOMER_TYPE		STRING,
	EVENT_EDITION_CODE		STRING,
	BADGE_UPDATE_FLG		STRING,
	MARKETING_PREF_PROMPT		STRING,
	ORG_NAME		STRING,
	ORG_CITY		STRING,
	ORG_COUNTRY		STRING,
	ORG_ID		STRING,
	ORG_STATE		STRING,
	PARTICIPATING_ORG_ID		STRING,
	PRODUCT_CODE		STRING,
	QR_CODE		STRING,
	REGISTRATION_ID		STRING,
	STATUS		BIGINT,
	STAFF_COMPANY_NAME		STRING,
	STAFF_COMPANY_ADDR		STRING,
	STAFF_PHONE_NUM		STRING,
	STAFF_REPORTING		STRING,
	STAFF_STANDS		STRING,
	STAFF_USER_ACCESS		STRING,
	VERSION_NUM		BIGINT,
	INTEGRATION_ID		STRING,
	DATASOURCE_NUM_ID		STRING,
	W_INSERT_DT		TIMESTAMP,
	W_UPDATE_DT		TIMESTAMP,
	MOBILEPHONE		STRING,
	FIRSTSCANNEDDATE		TIMESTAMP,
	LASTPRINTEDDATE		TIMESTAMP,
	FIRSTSCANNEDDATE_FLG		STRING,
	LASTPRINTEDDATE_FLG		STRING,
	ACCESSVALIDITYMODIFIEDDATE		TIMESTAMP,
	CREATEDDATE		TIMESTAMP,
	COMPANYPRODUCTCODE		STRING,
	PAYMENTSTATUS		STRING,
	PHOTOKEY		STRING,
	PHOTOSOURCE		STRING,
	PHOTOSOURCETYPE		STRING,
	PACKAGE_NAME		STRING,
	IND_UPDATE		STRING
)
USING DELTA;

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {100}: Insert into flow table (I$_)
INSERT INTO workspace.prxbi_dw.i_wc_badge_details_d_flow
(
	BADGE_ID,
	BADGE_LOCATION,
	BADGE_TOKEN,
	BADGE_VERSION,
	CONTACT_EMAIL,
	CONTACT_FIRST_NAME,
	CONTACT_LAST_NAME,
	CONTACT_JOB_TITLE,
	CONTACT_PERSON_ID,
	CREATION_REG_TYPE,
	CREATION_TYPE,
	CULTURE,
	CUSTOMER_TYPE,
	EVENT_EDITION_CODE,
	BADGE_UPDATE_FLG,
	MARKETING_PREF_PROMPT,
	ORG_NAME,
	ORG_CITY,
	ORG_COUNTRY,
	ORG_ID,
	ORG_STATE,
	PARTICIPATING_ORG_ID,
	PRODUCT_CODE,
	QR_CODE,
	REGISTRATION_ID,
	STATUS,
	STAFF_COMPANY_NAME,
	STAFF_COMPANY_ADDR,
	STAFF_PHONE_NUM,
	STAFF_REPORTING,
	STAFF_STANDS,
	STAFF_USER_ACCESS,
	VERSION_NUM,
	INTEGRATION_ID,
	DATASOURCE_NUM_ID,
	MOBILEPHONE,
	FIRSTSCANNEDDATE,
	LASTPRINTEDDATE,
	FIRSTSCANNEDDATE_FLG,
	LASTPRINTEDDATE_FLG,
	ACCESSVALIDITYMODIFIEDDATE,
	CREATEDDATE,
	COMPANYPRODUCTCODE,
	PAYMENTSTATUS,
	PHOTOKEY,
	PHOTOSOURCE,
	PHOTOSOURCETYPE,
	PACKAGE_NAME,
	IND_UPDATE
)
SELECT
    S.BADGE_ID,
    S.BADGE_LOCATION,
    S.BADGE_TOKEN,
    S.BADGE_VERSION,
    S.CONTACT_EMAIL,
    S.CONTACT_FIRST_NAME,
    S.CONTACT_LAST_NAME,
    S.CONTACT_JOB_TITLE,
    S.CONTACT_PERSON_ID,
    S.CREATION_REG_TYPE,
    S.CREATION_TYPE,
    S.CULTURE,
    S.CUSTOMER_TYPE,
    S.EVENT_EDITION_CODE,
    S.BADGE_UPDATE_FLG,
    S.MARKETING_PREF_PROMPT,
    S.ORG_NAME,
    S.ORG_CITY,
    S.ORG_COUNTRY,
    S.ORG_ID,
    S.ORG_STATE,
    S.PARTICIPATING_ORG_ID,
    S.PRODUCT_CODE,
    S.QR_CODE,
    S.REGISTRATION_ID,
    S.STATUS,
    S.STAFF_COMPANY_NAME,
    S.STAFF_COMPANY_ADDR,
    S.STAFF_PHONE_NUM,
    S.STAFF_REPORTING,
    S.STAFF_STANDS,
    S.STAFF_USER_ACCESS,
    S.VERSION_NUM,
    S.INTEGRATION_ID,
    S.DATASOURCE_NUM_ID,
    S.MOBILEPHONE,
    S.FIRSTSCANNEDDATE,
    S.LASTPRINTEDDATE,
    S.FIRSTSCANNEDDATE_FLG,
    S.LASTPRINTEDDATE_FLG,
    S.ACCESSVALIDITYMODIFIEDDATE,
    S.CREATEDDATE,
    S.COMPANYPRODUCTCODE,
    S.PAYMENTSTATUS,
    S.PHOTOKEY,
    S.PHOTOSOURCE,
    S.PHOTOSOURCETYPE,
    S.PACKAGE_NAME,
    S.IND_UPDATE
FROM (
    SELECT
        JOIN1_A.ID AS BADGE_ID,
        JOIN1_A.BADGELOCATION AS BADGE_LOCATION,
        JOIN1_A.BADGETOKEN AS BADGE_TOKEN,
        JOIN1_A.BADGEVERSION AS BADGE_VERSION,
        JOIN1_A.CONTACTEMAIL AS CONTACT_EMAIL,
        JOIN1_A.CONTACTFIRSTNAME AS CONTACT_FIRST_NAME,
        JOIN1_A.CONTACTLASTNAME AS CONTACT_LAST_NAME,
        JOIN1_A.CONTACTJOBTITLE AS CONTACT_JOB_TITLE,
        JOIN1_A.CONTACTPERSONRXMASTERID AS CONTACT_PERSON_ID,
        JOIN1_A.CREATEDBYREGISTRATIONTYPE AS CREATION_REG_TYPE,
        JOIN1_A.CREATEDBYTYPE AS CREATION_TYPE,
        JOIN1_A.CULTURE AS CULTURE,
        JOIN1_A.CUSTOMERTYPE AS CUSTOMER_TYPE,
        JOIN1_A.EVENTEDITIONGBSCODE AS EVENT_EDITION_CODE,
        JOIN1_A.ISBADGEUPDATE AS BADGE_UPDATE_FLG,
        JOIN1_A.MARKETINGPREFERENCESPROMPTREQU AS MARKETING_PREF_PROMPT,
        JOIN1_A.ORGANISATIONDISPLAYNAME AS ORG_NAME,
        JOIN1_A.ORGANISATIONCITY AS ORG_CITY,
        JOIN1_A.ORGANISATIONCOUNTRYCODE AS ORG_COUNTRY,
        JOIN1_A.ORGANISATIONRXMASTERID AS ORG_ID,
        JOIN1_A.ORGANISATIONSTATE AS ORG_STATE,
        JOIN1_A.PARTICIPATINGORGANISATIONID AS PARTICIPATING_ORG_ID,
        JOIN1_A.PRODUCTCODE AS PRODUCT_CODE,
        JOIN1_A.QRCODECONTENT AS QR_CODE,
        JOIN1_A.REGISTRATIONID AS REGISTRATION_ID,
        JOIN1_A.STATUS AS STATUS,
        JOIN1_A.SUPPORTSTAFFCOMPANYNAME AS STAFF_COMPANY_NAME,
        JOIN1_A.SUPPORTSTAFFCOMPANYADDRESS AS STAFF_COMPANY_ADDR,
        JOIN1_A.SUPPORTSTAFFMOBILEPHONE AS STAFF_PHONE_NUM,
        JOIN1_A.SUPPORTSTAFFREPORTSTO AS STAFF_REPORTING,
        JOIN1_A.SUPPORTSTAFFSTANDS AS STAFF_STANDS,
        JOIN1_A.SUPPORTSTAFFUSERACCESS AS STAFF_USER_ACCESS,
        JOIN1_A.VERSIONNUMBER AS VERSION_NUM,
        JOIN1_A.ID AS INTEGRATION_ID,
        CAST(${DATASOURCE_NUM_ID} AS STRING) AS DATASOURCE_NUM_ID,
        JOIN1_A.MOBILEPHONE AS MOBILEPHONE,
        JOIN1_A.FIRSTSCANNEDDATE AS FIRSTSCANNEDDATE,
        JOIN1_A.LASTPRINTEDDATE AS LASTPRINTEDDATE,
        CASE WHEN JOIN1_A.FIRSTSCANNEDDATE IS NOT NULL THEN 'Y' ELSE 'N' END AS FIRSTSCANNEDDATE_FLG,
        CASE WHEN JOIN1_A.LASTPRINTEDDATE IS NOT NULL THEN 'Y' ELSE 'N' END AS LASTPRINTEDDATE_FLG,
        JOIN1_A.ACCESSVALIDITYMODIFIEDDATE AS ACCESSVALIDITYMODIFIEDDATE,
        JOIN1_A.CREATEDDATE AS CREATEDDATE,
        JOIN1_A.COMPANYPRODUCTCODE AS COMPANYPRODUCTCODE,
        JOIN1_A.PAYMENTSTATUS AS PAYMENTSTATUS,
        JOIN1_A.PHOTOKEY AS PHOTOKEY,
        JOIN1_A.PHOTOSOURCE AS PHOTOSOURCE,
        JOIN1_A.PHOTOSOURCETYPE AS PHOTOSOURCETYPE,
        WC_BADGE_PRODUCT_D_2.NAME_1 AS PACKAGE_NAME,
        'I' AS IND_UPDATE
    FROM workspace.prxbi_dw.c_badge_details_stg AS JOIN1_A
    LEFT OUTER JOIN (
        SELECT
            WC_BADGE_PRODUCT_D_1.ID AS ID,
            WC_BADGE_PRODUCT_D_1.SKU AS SKU,
            WC_BADGE_PRODUCT_D_1.NAME AS NAME,
            WC_BADGE_PRODUCT_D_1.COL AS COL,
            WC_BADGE_PRODUCT_D_1.SKU_1 AS SKU_1,
            WC_BADGE_PRODUCT_D_1.NAME_1 AS NAME_1
        FROM (
            SELECT
                WC_BADGE_PRODUCT_D.ID AS ID,
                WC_BADGE_PRODUCT_D.SKU AS SKU,
                WC_BADGE_PRODUCT_D.NAME AS NAME,
                ROW_NUMBER() OVER(PARTITION BY WC_BADGE_PRODUCT_D.SKU ORDER BY WC_BADGE_PRODUCT_D.ID DESC) AS COL,
                WC_BADGE_PRODUCT_D.SKU AS SKU_1,
                WC_BADGE_PRODUCT_D.NAME AS NAME_1
            FROM workspace.prxbi_dw.wc_badge_product_d AS WC_BADGE_PRODUCT_D
        ) AS WC_BADGE_PRODUCT_D_1
        WHERE WC_BADGE_PRODUCT_D_1.COL = 1
    ) AS WC_BADGE_PRODUCT_D_2
    ON JOIN1_A.PRODUCTCODE = WC_BADGE_PRODUCT_D_2.SKU_1
) AS S
WHERE NOT EXISTS
	(
	SELECT 1 FROM workspace.prxbi_dw.wc_badge_details_d AS T
	WHERE T.INTEGRATION_ID <=> S.INTEGRATION_ID
	AND T.DATASOURCE_NUM_ID <=> S.DATASOURCE_NUM_ID
	AND T.BADGE_ID <=> S.BADGE_ID
	AND T.BADGE_LOCATION <=> S.BADGE_LOCATION
	AND T.BADGE_TOKEN <=> S.BADGE_TOKEN
	AND T.BADGE_VERSION <=> S.BADGE_VERSION
	AND T.CONTACT_EMAIL <=> S.CONTACT_EMAIL
	AND T.CONTACT_FIRST_NAME <=> S.CONTACT_FIRST_NAME
	AND T.CONTACT_LAST_NAME <=> S.CONTACT_LAST_NAME
	AND T.CONTACT_JOB_TITLE <=> S.CONTACT_JOB_TITLE
	AND T.CONTACT_PERSON_ID <=> S.CONTACT_PERSON_ID
	AND T.CREATION_REG_TYPE <=> S.CREATION_REG_TYPE
	AND T.CREATION_TYPE <=> S.CREATION_TYPE
	AND T.CULTURE <=> S.CULTURE
	AND T.CUSTOMER_TYPE <=> S.CUSTOMER_TYPE
	AND T.EVENT_EDITION_CODE <=> S.EVENT_EDITION_CODE
	AND T.BADGE_UPDATE_FLG <=> S.BADGE_UPDATE_FLG
	AND T.MARKETING_PREF_PROMPT <=> S.MARKETING_PREF_PROMPT
	AND T.ORG_NAME <=> S.ORG_NAME
	AND T.ORG_CITY <=> S.ORG_CITY
	AND T.ORG_COUNTRY <=> S.ORG_COUNTRY
	AND T.ORG_ID <=> S.ORG_ID
	AND T.ORG_STATE <=> S.ORG_STATE
	AND T.PARTICIPATING_ORG_ID <=> S.PARTICIPATING_ORG_ID
	AND T.PRODUCT_CODE <=> S.PRODUCT_CODE
	AND T.QR_CODE <=> S.QR_CODE
	AND T.REGISTRATION_ID <=> S.REGISTRATION_ID
	AND T.STATUS <=> S.STATUS
	AND T.STAFF_COMPANY_NAME <=> S.STAFF_COMPANY_NAME
	AND T.STAFF_COMPANY_ADDR <=> S.STAFF_COMPANY_ADDR
	AND T.STAFF_PHONE_NUM <=> S.STAFF_PHONE_NUM
	AND T.STAFF_REPORTING <=> S.STAFF_REPORTING
	AND T.STAFF_STANDS <=> S.STAFF_STANDS
	AND T.STAFF_USER_ACCESS <=> S.STAFF_USER_ACCESS
	AND T.VERSION_NUM <=> S.VERSION_NUM
	AND T.MOBILEPHONE <=> S.MOBILEPHONE
	AND T.FIRSTSCANNEDDATE <=> S.FIRSTSCANNEDDATE
	AND T.LASTPRINTEDDATE <=> S.LASTPRINTEDDATE
	AND T.FIRSTSCANNEDDATE_FLG <=> S.FIRSTSCANNEDDATE_FLG
	AND T.LASTPRINTEDDATE_FLG <=> S.LASTPRINTEDDATE_FLG
	AND T.ACCESSVALIDITYMODIFIEDDATE <=> S.ACCESSVALIDITYMODIFIEDDATE
	AND T.CREATEDDATE <=> S.CREATEDDATE
	AND T.COMPANYPRODUCTCODE <=> S.COMPANYPRODUCTCODE
	AND T.PAYMENTSTATUS <=> S.PAYMENTSTATUS
	AND T.PHOTOKEY <=> S.PHOTOKEY
	AND T.PHOTOSOURCE <=> S.PHOTOSOURCE
	AND T.PHOTOSOURCETYPE <=> S.PHOTOSOURCETYPE
	AND T.PACKAGE_NAME <=> S.PACKAGE_NAME
	);

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {110}:
Create index
on flow table -> Converted to
OPTIMIZE ZORDER BY
-- Disable
ZORDER stats check to prevent DELTA_ZORDERING_ON_COLUMN_WITHOUT_STATS
SET spark.databricks.delta.optimize.zorder.checkStatsCollection.enabled = false;
OPTIMIZE workspace.prxbi_dw.i_wc_badge_details_d_flow
ZORDER BY (INTEGRATION_ID, DATASOURCE_NUM_ID);

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {120}: DBMS_STATS.GATHER_TABLE_STATS is not applicable for Delta flow tables, removed.
SELECT COUNT(*)
FROM workspace.prxbi_dw.i_wc_badge_details_d_flow;

## Mark Records for Update

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {130}: Mark records in flow table for update ('U') if they exist in target
-- Converted Oracle tuple-IN UPDATE to MERGE
MERGE INTO workspace.prxbi_dw.i_wc_badge_details_d_flow AS T
USING (
    SELECT
        INTEGRATION_ID,
        DATASOURCE_NUM_ID
    FROM workspace.prxbi_dw.wc_badge_details_d
) AS S
ON T.INTEGRATION_ID = S.INTEGRATION_ID
AND T.DATASOURCE_NUM_ID = S.DATASOURCE_NUM_ID
WHEN MATCHED THEN UPDATE SET T.IND_UPDATE = 'U';

## MERGE into Target (`WC_BADGE_DETAILS_D`)

In [ ]:
-- MAGIC %sql
-- Ensure target table schema for WC_BADGE_DETAILS_D exists, assuming ROW_WID is GENERATED ALWAYS AS IDENTITY
-- This DDL would typically be in a separate, preceding DDL notebook or already existing.
-- For demonstration purposes, including a basic CREATE TABLE IF NOT EXISTS here.
CREATE TABLE IF NOT EXISTS workspace.prxbi_dw.wc_badge_details_d (
    ROW_WID BIGINT GENERATED ALWAYS AS IDENTITY,
    BADGE_ID STRING,
    BADGE_LOCATION STRING,
    BADGE_TOKEN STRING,
    BADGE_VERSION SMALLINT,
    CONTACT_EMAIL STRING,
    CONTACT_FIRST_NAME STRING,
    CONTACT_LAST_NAME STRING,
    CONTACT_JOB_TITLE STRING,
    CONTACT_PERSON_ID STRING,
    CREATION_REG_TYPE STRING,
    CREATION_TYPE STRING,
    CULTURE STRING,
    CUSTOMER_TYPE STRING,
    EVENT_EDITION_CODE STRING,
    BADGE_UPDATE_FLG STRING,
    MARKETING_PREF_PROMPT STRING,
    ORG_NAME STRING,
    ORG_CITY STRING,
    ORG_COUNTRY STRING,
    ORG_ID STRING,
    ORG_STATE STRING,
    PARTICIPATING_ORG_ID STRING,
    PRODUCT_CODE STRING,
    QR_CODE STRING,
    REGISTRATION_ID STRING,
    STATUS BIGINT,
    STAFF_COMPANY_NAME STRING,
    STAFF_COMPANY_ADDR STRING,
    STAFF_PHONE_NUM STRING,
    STAFF_REPORTING STRING,
    STAFF_STANDS STRING,
    STAFF_USER_ACCESS STRING,
    VERSION_NUM BIGINT,
    INTEGRATION_ID STRING,
    DATASOURCE_NUM_ID STRING,
    W_INSERT_DT TIMESTAMP,
    W_UPDATE_DT TIMESTAMP,
    MOBILEPHONE STRING,
    FIRSTSCANNEDDATE TIMESTAMP,
    LASTPRINTEDDATE TIMESTAMP,
    FIRSTSCANNEDDATE_FLG STRING,
    LASTPRINTEDDATE_FLG STRING,
    ACCESSVALIDITYMODIFIEDDATE TIMESTAMP,
    CREATEDDATE TIMESTAMP,
    COMPANYPRODUCTCODE STRING,
    PAYMENTSTATUS STRING,
    PHOTOKEY STRING,
    PHOTOSOURCE STRING,
    PHOTOSOURCETYPE STRING,
    PACKAGE_NAME STRING
) USING DELTA;

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {150} and {160} combined: Merge into target dimension table
-- Converted Oracle tuple-SET UPDATE and separate INSERT into a single MERGE statement.
-- ROW_WID is excluded from INSERT list, assuming GENERATED ALWAYS AS IDENTITY.
MERGE INTO workspace.prxbi_dw.wc_badge_details_d AS T
USING workspace.prxbi_dw.i_wc_badge_details_d_flow AS S
ON T.INTEGRATION_ID = S.INTEGRATION_ID
AND T.DATASOURCE_NUM_ID = S.DATASOURCE_NUM_ID
WHEN MATCHED AND S.IND_UPDATE = 'U' THEN UPDATE SET
    T.BADGE_ID = S.BADGE_ID,
    T.BADGE_LOCATION = S.BADGE_LOCATION,
    T.BADGE_TOKEN = S.BADGE_TOKEN,
    T.BADGE_VERSION = S.BADGE_VERSION,
    T.CONTACT_EMAIL = S.CONTACT_EMAIL,
    T.CONTACT_FIRST_NAME = S.CONTACT_FIRST_NAME,
    T.CONTACT_LAST_NAME = S.CONTACT_LAST_NAME,
    T.CONTACT_JOB_TITLE = S.CONTACT_JOB_TITLE,
    T.CONTACT_PERSON_ID = S.CONTACT_PERSON_ID,
    T.CREATION_REG_TYPE = S.CREATION_REG_TYPE,
    T.CREATION_TYPE = S.CREATION_TYPE,
    T.CULTURE = S.CULTURE,
    T.CUSTOMER_TYPE = S.CUSTOMER_TYPE,
    T.EVENT_EDITION_CODE = S.EVENT_EDITION_CODE,
    T.BADGE_UPDATE_FLG = S.BADGE_UPDATE_FLG,
    T.MARKETING_PREF_PROMPT = S.MARKETING_PREF_PROMPT,
    T.ORG_NAME = S.ORG_NAME,
    T.ORG_CITY = S.ORG_CITY,
    T.ORG_COUNTRY = S.ORG_COUNTRY,
    T.ORG_ID = S.ORG_ID,
    T.ORG_STATE = S.ORG_STATE,
    T.PARTICIPATING_ORG_ID = S.PARTICIPATING_ORG_ID,
    T.PRODUCT_CODE = S.PRODUCT_CODE,
    T.QR_CODE = S.QR_CODE,
    T.REGISTRATION_ID = S.REGISTRATION_ID,
    T.STATUS = S.STATUS,
    T.STAFF_COMPANY_NAME = S.STAFF_COMPANY_NAME,
    T.STAFF_COMPANY_ADDR = S.STAFF_COMPANY_ADDR,
    T.STAFF_PHONE_NUM = S.STAFF_PHONE_NUM,
    T.STAFF_REPORTING = S.STAFF_REPORTING,
    T.STAFF_STANDS = S.STAFF_STANDS,
    T.STAFF_USER_ACCESS = S.STAFF_USER_ACCESS,
    T.VERSION_NUM = S.VERSION_NUM,
    T.MOBILEPHONE = S.MOBILEPHONE,
    T.FIRSTSCANNEDDATE = S.FIRSTSCANNEDDATE,
    T.LASTPRINTEDDATE = S.LASTPRINTEDDATE,
    T.FIRSTSCANNEDDATE_FLG = S.FIRSTSCANNEDDATE_FLG,
    T.LASTPRINTEDDATE_FLG = S.LASTPRINTEDDATE_FLG,
    T.ACCESSVALIDITYMODIFIEDDATE = S.ACCESSVALIDITYMODIFIEDDATE,
    T.CREATEDDATE = S.CREATEDDATE,
    T.COMPANYPRODUCTCODE = S.COMPANYPRODUCTCODE,
    T.PAYMENTSTATUS = S.PAYMENTSTATUS,
    T.PHOTOKEY = S.PHOTOKEY,
    T.PHOTOSOURCE = S.PHOTOSOURCE,
    T.PHOTOSOURCETYPE = S.PHOTOSOURCETYPE,
    T.PACKAGE_NAME = S.PACKAGE_NAME,
    T.W_UPDATE_DT = current_timestamp()
WHEN NOT MATCHED AND S.IND_UPDATE = 'I' THEN INSERT
    (
    BADGE_ID,
    BADGE_LOCATION,
    BADGE_TOKEN,
    BADGE_VERSION,
    CONTACT_EMAIL,
    CONTACT_FIRST_NAME,
    CONTACT_LAST_NAME,
    CONTACT_JOB_TITLE,
    CONTACT_PERSON_ID,
    CREATION_REG_TYPE,
    CREATION_TYPE,
    CULTURE,
    CUSTOMER_TYPE,
    EVENT_EDITION_CODE,
    BADGE_UPDATE_FLG,
    MARKETING_PREF_PROMPT,
    ORG_NAME,
    ORG_CITY,
    ORG_COUNTRY,
    ORG_ID,
    ORG_STATE,
    PARTICIPATING_ORG_ID,
    PRODUCT_CODE,
    QR_CODE,
    REGISTRATION_ID,
    STATUS,
    STAFF_COMPANY_NAME,
    STAFF_COMPANY_ADDR,
    STAFF_PHONE_NUM,
    STAFF_REPORTING,
    STAFF_STANDS,
    STAFF_USER_ACCESS,
    VERSION_NUM,
    INTEGRATION_ID,
    DATASOURCE_NUM_ID,
    MOBILEPHONE,
    FIRSTSCANNEDDATE,
    LASTPRINTEDDATE,
    FIRSTSCANNEDDATE_FLG,
    LASTPRINTEDDATE_FLG,
    ACCESSVALIDITYMODIFIEDDATE,
    CREATEDDATE,
    COMPANYPRODUCTCODE,
    PAYMENTSTATUS,
    PHOTOKEY,
    PHOTOSOURCE,
    PHOTOSOURCETYPE,
    PACKAGE_NAME,
    W_INSERT_DT,
    W_UPDATE_DT
    )
VALUES
    (
    S.BADGE_ID,
    S.BADGE_LOCATION,
    S.BADGE_TOKEN,
    S.BADGE_VERSION,
    S.CONTACT_EMAIL,
    S.CONTACT_FIRST_NAME,
    S.CONTACT_LAST_NAME,
    S.CONTACT_JOB_TITLE,
    S.CONTACT_PERSON_ID,
    S.CREATION_REG_TYPE,
    S.CREATION_TYPE,
    S.CULTURE,
    S.CUSTOMER_TYPE,
    S.EVENT_EDITION_CODE,
    S.BADGE_UPDATE_FLG,
    S.MARKETING_PREF_PROMPT,
    S.ORG_NAME,
    S.ORG_CITY,
    S.ORG_COUNTRY,
    S.ORG_ID,
    S.ORG_STATE,
    S.PARTICIPATING_ORG_ID,
    S.PRODUCT_CODE,
    S.QR_CODE,
    S.REGISTRATION_ID,
    S.STATUS,
    S.STAFF_COMPANY_NAME,
    S.STAFF_COMPANY_ADDR,
    S.STAFF_PHONE_NUM,
    S.STAFF_REPORTING,
    S.STAFF_STANDS,
    S.STAFF_USER_ACCESS,
    S.VERSION_NUM,
    S.INTEGRATION_ID,
    S.DATASOURCE_NUM_ID,
    S.MOBILEPHONE,
    S.FIRSTSCANNEDDATE,
    S.LASTPRINTEDDATE,
    S.FIRSTSCANNEDDATE_FLG,
    S.LASTPRINTEDDATE_FLG,
    S.ACCESSVALIDITYMODIFIEDDATE,
    S.CREATEDDATE,
    S.COMPANYPRODUCTCODE,
    S.PAYMENTSTATUS,
    S.PHOTOKEY,
    S.PHOTOSOURCE,
    S.PHOTOSOURCETYPE,
    S.PACKAGE_NAME,
    current_timestamp(),
    current_timestamp()
    );

## Cleanup

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {180}:
Drop flow table
DROP TABLE IF EXISTS workspace.prxbi_dw.i_wc_badge_details_d_flow;

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {210}:
Drop staging table
DROP TABLE IF EXISTS workspace.prxbi_dw.c_badge_details_stg;

## Validation

In [ ]:
-- MAGIC %sql
SELECT COUNT(*) AS final_record_count
FROM workspace.prxbi_dw.wc_badge_details_d;

In [ ]:
-- MAGIC %sql
SELECT *
FROM workspace.prxbi_dw.wc_badge_details_d
ORDER BY W_UPDATE_DT DESC LIMIT 10;

## Conversion Notes and Manual Actions Required

1.  **ROW_WID for `wc_badge_details_d`:** The conversion assumes `ROW_WID` in `workspace.prxbi_dw.wc_badge_details_d` is a `GENERATED ALWAYS AS IDENTITY` column. If it is `GENERATED BY DEFAULT AS IDENTITY`, the `ROW_WID` column can be included in the `INSERT` list of the `MERGE` statement.
2.  **ETL Parameters:** `ETL_JOB_TYPE`, `DATASOURCE_NUM_ID`, `ETL_PROC_WID`, `ODI_SESS_NO` are created as Databricks widgets. Ensure these are set correctly before execution.
3.  **Target Table DDL:** A `CREATE TABLE IF NOT EXISTS` statement for `workspace.prxbi_dw.wc_badge_details_d` has been included for completeness. In a production environment, this table's DDL should ideally be managed separately and exist prior to the ETL process.
4.  **ODI Session Tasks:** `SCEN_TASK_NO {1}, {4}, {5}, {10}, {20}, {70}, {80} (second instance), {140}, {170}, {190}, {200}` were either duplicate, comment-only, or non-applicable ODI control statements and have been omitted or converted into comments within the nearest logical cell.